# Chess Analyzer — 端到端演示

本 Notebook 演示如何使用 `chess_analyzer` 包，从一份 PGN 文件出发，
完整走完「解析 → 特征提取 → 等级分预测 → 可视化」四个步骤。

> 提示：`examples/sample.pgn` 中的 5 盘对局是**模拟数据**（合法走法 + `%clk` 时间戳），
> 仅用于演示流程，不代表真实棋手的真实战绩。

## 1. 安装依赖

In [ ]:
%pip install -r ../requirements.txt

## 2. 配置初始化

所有阈值 / 路径都集中在 `configs/default.yaml`。不提供配置时，
`load_config()` 会自动使用包内置的默认值（与原脚本行为一致）。

如果你本地装了 Stockfish，把下面的 `stockfish.path` 改成你的可执行文件路径，
或者直接设置环境变量 `STOCKFISH_PATH`；不设置的话，引擎相关步骤会自动降级为
「仅廉价特征」模式（速度更快，但预测准确率约 50%）。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from chess_analyzer.core.config import load_config

cfg = load_config("../configs/default.yaml")
print("Stockfish 路径:", cfg.stockfish.path)
print("搜索深度:", cfg.stockfish.depth)
print("开局步数阈值:", cfg.thresholds.opening_moves)

# 如需覆盖，例如指向你本机装好的 Stockfish：
# os.environ["STOCKFISH_PATH"] = "/usr/local/bin/stockfish"
# cfg = load_config("../configs/default.yaml")  # 重新加载以应用环境变量覆盖

## 3. 运行特征提取（对已有 parquet 数据）

`run_feature_pipeline` 面向的是「已经跑过 curation.py + ACPL_new.py，
拿到 `{category}.parquet` / `{category}_gm_detailed_evals.parquet` /
`{category}_gm_aggregated_metrics.parquet` 三张表」之后的场景。

本 Notebook 用的是单个 PGN + 端到端推理（见第 4 步），这一步仅作 API 演示：
如果你已经有预处理好的数据目录，可以直接这样调用：

In [ ]:
from chess_analyzer.pipeline import run_feature_pipeline

# 示例（需要你自己准备好 classical.parquet 等输入数据）：
# result = run_feature_pipeline(
#     meta_path="classical/classical.parquet",
#     step_path="classical/classical_gm_detailed_evals.parquet",
#     agg_path="classical/classical_gm_aggregated_metrics.parquet",
#     output_dir="classical/style",
#     config=cfg,
# )
# result['style'].head()
print("run_feature_pipeline 已就绪，签名: run_feature_pipeline(meta_path, step_path, agg_path, output_dir, config)")

## 4. 运行预测：从 sample.pgn 到等级分区间

`run_inference` 是最贴近「拿到一份 PGN，问 AI 这个人大概什么水平」这个场景的入口。
`--no_engine` 效果对应下面的 `no_engine=True`：跳过 Stockfish，只用 16 维「廉价特征」
快速给出一个粗略区间（约 50% 准确率），适合在没有装 Stockfish 的环境里跑通全流程演示。

In [ ]:
from chess_analyzer.pipeline import run_inference

# 注：完整推理需要 `models/` 目录下已有训练好的第一层/第二层模型
# （由本项目之外的 train_stacked_model.py / train_lgb_layer2.py 产出）。
# 首次克隆仓库、尚未放入模型文件时，这里会给出提示而不是让 Notebook 崩溃。
try:
    result = run_inference(
        pgn_path="sample.pgn",
        name="Demo Player",
        data_dir="./_demo_output",
        no_engine=True,   # 改成 False 可启用 Stockfish 深度分析（需要本机装有 Stockfish）
        min_ply=10,       # sample.pgn 里的对局较短，演示用途调低下限
        config=cfg,
    )
    print(result["final_tier"], result["avg_proba"])
except SystemExit:
    print("⚠️ 未找到训练好的模型文件（models/ 目录）。请先运行本项目的模型训练脚本，"
          "或将已训练好的 .pkl / .txt 模型文件放入 configs/default.yaml 中 "
          "inference.model_dir 指向的目录后再运行本单元格。")

## 5. 可视化：风格雷达图 / 能力雷达图

完整的雷达图 / ACPL 分布图需要「目标棋手」和「基线人群」两组特征数据
（见 `run_style_report`，对应原 `extra_style.py`）。这里演示如何直接调用
底层绘图函数，把 `run_feature_pipeline` 产出的两组风格特征表画成图：


In [ ]:
import matplotlib.pyplot as plt
from chess_analyzer.viz import plot_style_and_opening

# 示例（需要准备好 player_style / baseline_style 两个 DataFrame）：
# style_percentiles = plot_style_and_opening(
#     player_style, baseline_style,
#     player_opening=None, baseline_opening=None,
#     player_name="Demo Player",
#     save_path="./_demo_output/style_radar.png",
# )
# from IPython.display import Image
# Image("./_demo_output/style_radar.png")
print("plot_style_and_opening 已就绪，会在 save_path 生成 PNG 并可用 IPython.display.Image 直接渲染。")

## 6. 一次性生成完整报告（推荐入口）

如果你已经有「目标棋手」和「基线人群（Lichess / TWIC）」两套预处理好的
风格 + 分项能力表，`run_style_report` 会一次性生成：
风格+开局图、三张能力雷达图、ACPL 分布图、百分位汇总 CSV、风格诊断文本。

In [ ]:
from chess_analyzer.pipeline import run_style_report

# report = run_style_report(config=cfg, export_ml=False)
print("run_style_report 已就绪，签名: run_style_report(config=None, export_ml=False)")